# 01 Carga y Validacion de Parquet

Este notebook consulta el `OLAP` de Victor, reconstruye la base por ticket y genera el `Parquet` principal de la etapa `01`.


## 1. Librerias


In [1]:
# Manejo de rutas internas del proyecto.
from pathlib import Path
# Captura de salida de texto para resumir resultados como df.info().
import io
# Lectura de variables de entorno para la conexion.
import os
# Control de advertencias durante la lectura con pandas.
import warnings

# Manejo principal de datos tabulares.
import pandas as pd
# Conexion directa a PostgreSQL.
import psycopg2


### Para que se usan estas librerias

- `Path`: localizar la raiz del proyecto y construir rutas de entrada y salida.
- `io`: capturar salidas como `df.info()` en formato de texto para documentarlas mejor.
- `os`: leer variables de entorno y definir la conexion sin dejarla fija solo en una ruta.
- `warnings`: controlar advertencias que no afectan el resultado del notebook.
- `pandas`: leer, validar, revisar y exportar la base a `Parquet`.
- `psycopg2`: conectarse a PostgreSQL y consultar el esquema `olap`.


## 2. Definir rutas y conexion


In [2]:
from pathlib import Path

def resolve_project_root() -> Path:
    # Buscar la raiz del proyecto a partir del directorio actual.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "parquets").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto")

PROJECT_ROOT = resolve_project_root()

# Definir la ruta del SQL base y la salida principal de la etapa 01.
SQL_PATH = PROJECT_ROOT / "sql" / "04_base_tickets_modelado.sql"
OUTPUT_DIR = PROJECT_ROOT / "parquets" / "01_Carga_y_Validacion_Parquet"
OUTPUT_PATH = OUTPUT_DIR / "01_base_tickets_modelado.parquet"

# Definir la conexion local a PostgreSQL.
CONNECTION_PARAMS = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": os.getenv("PGPORT", "5432"),
    "dbname": os.getenv("PGDATABASE", "restaurante"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "postgres"),
}

print(f"Raiz del proyecto: {PROJECT_ROOT}")
print(f"SQL de entrada: {SQL_PATH}")
print(f"Parquet de salida: {OUTPUT_PATH}")


Raiz del proyecto: D:\08_Octavo semestre La Salle\Desarrollo De Inteligencia De Negocios\PROYECTO Maching learing\Capa Machine Learning
SQL de entrada: D:\08_Octavo semestre La Salle\Desarrollo De Inteligencia De Negocios\PROYECTO Maching learing\Capa Machine Learning\sql\04_base_tickets_modelado.sql
Parquet de salida: D:\08_Octavo semestre La Salle\Desarrollo De Inteligencia De Negocios\PROYECTO Maching learing\Capa Machine Learning\parquets\01_Carga_y_Validacion_Parquet\01_base_tickets_modelado.parquet


## 3. Cargar la consulta y leer la base desde OLAP


In [3]:
# Leer la consulta que reconstruye la base de tickets.
query = SQL_PATH.read_text(encoding="utf-8")

# Ejecutar la consulta en PostgreSQL y cargar el resultado en un DataFrame.
connection = psycopg2.connect(**CONNECTION_PARAMS)
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy connectable")
        df = pd.read_sql_query(query, connection)
finally:
    connection.close()

# Mostrar las primeras filas de la base reconstruida.
df.head()


,id_ticket_modelado,id_tiempo,fecha,dia,mes,nombre_mes,trimestre,anio,dia_semana,fin_semana,...,categorias_distintas,subtotal_ticket,total_pedido,monto_pago,diferencia_pago,incluye_bebida,incluye_postre,incluye_entrada,incluye_platillo_fuerte,ticket_alto
0,1,1,2024-01-02,2,1,Enero,1,2024,Martes,False,...,3,945.0,250.0,250.0,0.0,0,0,0,1,0
1,2,2,2024-01-03,3,1,Enero,1,2024,Miércoles,False,...,10,3865.0,285.0,285.0,0.0,1,1,1,1,0
2,3,2,2024-01-03,3,1,Enero,1,2024,Miércoles,False,...,10,5170.0,420.0,420.0,0.0,1,1,1,1,0
3,4,2,2024-01-03,3,1,Enero,1,2024,Miércoles,False,...,5,1485.0,370.0,305.0,-65.0,1,0,0,1,0
4,5,2,2024-01-03,3,1,Enero,1,2024,Miércoles,False,...,1,210.0,335.0,335.0,0.0,0,0,0,1,0


### Resultado

La consulta se ejecuto correctamente sobre el esquema `olap` y devolvio una vista previa de `5 filas x 36 columnas`.

En las primeras filas ya se observan variables importantes para el proyecto, como:

- `fecha`
- `ciudad`
- `metodo_pago`
- `categorias_distintas`
- `subtotal_ticket`
- `total_pedido`
- `monto_pago`
- `ticket_alto`

En esta salida inicial tambien se observa que la base ya quedo reconstruida por ticket y no por linea individual de venta.


## 4. Validar dimensiones y tipos


In [4]:
# Revisar la cantidad de filas y columnas generadas.
df.shape


(1167, 36)

In [5]:
# Revisar tipos de datos de forma compacta.
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1167 entries, 0 to 1166
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id_ticket_modelado       1167 non-null   int64  
 1   id_tiempo                1167 non-null   int64  
 2   fecha                    1167 non-null   object 
 3   dia                      1167 non-null   int64  
 4   mes                      1167 non-null   int64  
 5   nombre_mes               1167 non-null   object 
 6   trimestre                1167 non-null   int64  
 7   anio                     1167 non-null   int64  
 8   dia_semana               1167 non-null   object 
 9   fin_semana               1167 non-null   bool   
 10  id_cliente               1167 non-null   int64  
 11  id_sucursal              1167 non-null   int64  
 12  ciudad                   1167 non-null   object 
 13  capacidad_sucursal       1167 non-null   int64  
 14  id_empleado             

### Resultado

La validacion de estructura confirmo que la base quedo con `1167 filas` y `36 columnas`.

Tambien se verifico que:

- todas las columnas quedaron con `1167` valores no nulos
- no se detectaron faltantes en esta etapa
- los tipos quedaron repartidos en `bool`, `float64`, `int64` y `object`

Esto indica que la base reconstruida es estable y puede continuar al proceso de validacion de negocio y exportacion.


## 5. Revisar nulos y columnas clave


In [6]:
# Revisar cuantos valores nulos existen por columna.
df.isna().sum().sort_values(ascending=False).head(15)


id_ticket_modelado    0
id_tiempo             0
fecha                 0
dia                   0
mes                   0
nombre_mes            0
trimestre             0
anio                  0
dia_semana            0
fin_semana            0
id_cliente            0
id_sucursal           0
ciudad                0
capacidad_sucursal    0
id_empleado           0
dtype: int64

In [7]:
# Revisar rapidamente las columnas clave del negocio.
df[["fecha", "ciudad", "metodo_pago", "total_pedido", "monto_pago", "ticket_alto"]].head(10)


,fecha,ciudad,metodo_pago,total_pedido,monto_pago,ticket_alto
0,2024-01-02,León,Efectivo,250.0,250.0,0
1,2024-01-03,León,Efectivo,285.0,285.0,0
2,2024-01-03,León,Tarjeta de crédito,420.0,420.0,0
3,2024-01-03,León,Efectivo,370.0,305.0,0
4,2024-01-03,Guanajuato,Efectivo,335.0,335.0,0
5,2024-01-04,León,Efectivo,195.0,195.0,0
6,2024-01-04,León,Tarjeta de débito,560.0,560.0,1
7,2024-01-04,León,Efectivo,340.0,340.0,0
8,2024-01-04,León,Efectivo,345.0,345.0,0
9,2024-01-05,León,Efectivo,310.0,310.0,0


### Resultado

La revision de nulos mostro `0` faltantes en las columnas clave revisadas.

En la muestra de negocio ya se alcanzan a ver datos utiles del restaurante, por ejemplo:

- ciudades como `Leon` y `Guanajuato`
- metodos de pago como `Efectivo`, `Tarjeta de credito` y `Tarjeta de debito`
- valores de `total_pedido` y `monto_pago`
- la etiqueta `ticket_alto`, donde ya se observa al menos un caso marcado con `1`

Esto confirma que la base ya contiene variables de negocio listas para las etapas siguientes.


## 6. Exportar el parquet de la etapa 01


In [8]:
# Crear la carpeta de salida si todavia no existe.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exportar la base principal del proyecto a formato Parquet.
df.to_parquet(OUTPUT_PATH, index=False)

# Confirmar la ruta de salida generada.
print(f"Parquet generado en: {OUTPUT_PATH}")


Parquet generado en: D:\08_Octavo semestre La Salle\Desarrollo De Inteligencia De Negocios\PROYECTO Maching learing\Capa Machine Learning\parquets\01_Carga_y_Validacion_Parquet\01_base_tickets_modelado.parquet


In [9]:
# Volver a leer el parquet exportado para verificar que se guardo correctamente.
df_exportado = pd.read_parquet(OUTPUT_PATH)
df_exportado.shape


(1167, 36)

### Resultado

El archivo `01_base_tickets_modelado.parquet` se genero correctamente en la ruta esperada dentro de la carpeta `parquets/01_Carga_y_Validacion_Parquet`.

Despues de guardarlo, se volvio a leer y se confirmo que conserva la misma estructura:

- `1167 filas`
- `36 columnas`

Con esto, la etapa `01` queda cerrada y lista para alimentar la etapa `02` del `EDA`.


## Conclusi?n

La etapa `01` reconstruye la base principal desde el OLAP y la deja guardada como `01_base_tickets_modelado.parquet` con `1167 filas` y `36 columnas`. Con esto se establece la base por ticket que alimenta el resto del proyecto.
